# 在 Kaggle 上训练两个 Mamba-3 MiniGrid 记忆智能体 (T4 16GB 极限版)

此版本针对 **NVIDIA T4 16GB** 和 **Mamba-3** 架构进行了极限优化，同时并行训练两个实验：
- **实验 A**: MemoryS11 → 快速验证 Mamba-3 是否学到了记忆
- **实验 B**: MemoryS17Random → 真正的长序列记忆挑战

推荐的 Kaggle 设置：
- **加速器**: GPU T4 x2 (两个实验各占一块 GPU)
- **互联网**: 开启
- **Mamba 版本**: 2.3.2.post1+ (包含 Mamba-3)
- **预计时间**: 12 小时 Kaggle session
</cell_id>


In [ ]:
# ===== 用户配置 (T4 16GB + Mamba-3 极限版) =====
from pathlib import Path
import os, json, time

REPO_URL = "https://github.com/SShion0721/mamba-minigrid-memory.git"
REPO_BRANCH = "main"

WORKDIR = Path("/kaggle/working/mamba-minigrid-memory")
RUNS_DIR = WORKDIR / "runs"

# ---- 实验 A: MemoryS11 快速验证 ----
EXP_A = dict(
    name="mamba3_s11_t4_seed42",
    env_id="MiniGrid-MemoryS11-v0",
    total_steps=5_000_000,
    num_envs=128,          # T4 16GB: 极限并行度
    num_steps=256,         # 长 rollout 提升 sample efficiency
    context_len=128,       # 长记忆窗口
    chunk_len=128,         # 匹配 context_len
    batch_chunks=64,       # T4 16GB: 大批量 chunk 训练
    d_model=256,           # 大模型容量
    mamba_layers=6,        # 深层 Mamba
    d_state=16,
    seed=42,
)

# ---- 实验 B: MemoryS17Random 终极挑战 ----
EXP_B = dict(
    name="mamba3_s17random_t4_seed42",
    env_id="MiniGrid-MemoryS17Random-v0",
    total_steps=10_000_000, # S17Random 需要更多样本
    num_envs=128,
    num_steps=256,
    context_len=128,
    chunk_len=128,
    batch_chunks=64,
    d_model=256,
    mamba_layers=6,
    d_state=16,
    seed=42,
)

# ---- 共享 Mamba-3 参数 ----
VARIANT = "mamba3"
SPATIAL_ENCODER = "hybrid"
SPATIAL_LAYERS = 2
SPATIAL_HEADS = 4
VALID_ACTIONS = "0,1,2"

# ---- PPO 超参数 ----
LR = 1e-4                # Mamba-3 用更低学习率更稳
ENT_COEF = 0.01
ENT_COEF_FINAL = 0.001
CLIP_COEF = 0.2
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5
GAMMA = 0.99
GAE_LAMBDA = 0.95

# ---- 评估与保存 ----
EVAL_INTERVAL = 50_000
EVAL_EPISODES = 30
SAVE_INTERVAL = 200_000
LOG_INTERVAL = 5         # 更频繁日志便于监控

# GPU 分配
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")


In [ ]:
# ===== 硬件与环境检查 =====
import subprocess, sys

def run(cmd, check=True, cwd=None, env=None):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)

run("nvidia-smi", check=False)
print(f"Python 版本: {sys.version}")
print(f"可见 GPU 数量: {len(os.environ.get('CUDA_VISIBLE_DEVICES', '0,1').split(','))}")


In [ ]:
# ===== 安装最新依赖 (适配 Mamba-3) =====
run(f"{sys.executable} -m pip install -q --upgrade pip")
run(f"{sys.executable} -m pip install -q gymnasium minigrid tensorboard tqdm einops imageio[ffmpeg]")

print("正在安装最新版 mamba-ssm (包含 Mamba-3)...")
# 注意: Kaggle 环境可能已有 PyTorch CUDA 版, 不要重装 torch
run(f"{sys.executable} -m pip install -q --no-cache-dir causal-conv1d mamba-ssm --no-build-isolation")

import torch
import numpy as np
print(f"torch: {torch.__version__}, cuda: {torch.version.cuda}, devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}, 显存: {torch.cuda.get_device_properties(i).total_mem/1e9:.1f} GB")
print(f"numpy: {np.__version__}")

# 验证 Mamba-3 可用
try:
    from mamba_ssm import Mamba3
    print(f"✓ Mamba3 导入成功")
except ImportError:
    from mamba_ssm.modules.mamba3 import Mamba3
    print(f"✓ Mamba3 导入成功 (secondary path)")

# 验证 hybrid encoder 依赖
from mamba_ssm import Mamba as MambaBlock
print(f"✓ Mamba1 导入成功 (用于兼容性)")
print(f"\n✓ 所有依赖就绪")


In [ ]:
# ===== 获取项目代码 =====
import shutil, sys
sys.path.insert(0, str(WORKDIR))

if WORKDIR.exists():
    print("清理旧代码...")
    shutil.rmtree(WORKDIR)

run(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {WORKDIR}")
assert (WORKDIR / "src" / "train_mamba_ppo.py").exists(), "train_mamba_ppo.py 不存在!"
assert (WORKDIR / "src" / "models.py").exists(), "models.py 不存在!"
print("✓ 项目代码就绪")


In [ ]:
# ===== 启动两个 Mamba-3 并行实验 (T4 16GB 极限版) =====
#
# GPU 分配:
#   - 实验 A (S11):  GPU 0
#   - 实验 B (S17Random): GPU 1
#
# 如果只有一个 GPU, 两个实验会串行执行。

import threading, subprocess, sys, time, os
from pathlib import Path

WORKDIR = Path("/kaggle/working/mamba-minigrid-memory")
os.chdir(WORKDIR)

def build_training_cmd(exp: dict, gpu_id: int) -> list[str]:
    """构建训练命令行参数"""
    cmd = [
        sys.executable, "-u", str(WORKDIR / "src" / "train_mamba_ppo.py"),
        "--model", "mamba",
        "--mamba-variant", VARIANT,
        "--env-id", exp["env_id"],
        "--seed", str(exp["seed"]),
        "--total-steps", str(exp["total_steps"]),
        "--num-envs", str(exp["num_envs"]),
        "--num-steps", str(exp["num_steps"]),
        "--context-len", str(exp["context_len"]),
        "--chunk-len", str(exp["chunk_len"]),
        "--batch-chunks", str(exp["batch_chunks"]),
        "--d-model", str(exp["d_model"]),
        "--mamba-layers", str(exp["mamba_layers"]),
        "--d-state", str(exp["d_state"]),
        "--spatial-encoder", SPATIAL_ENCODER,
        "--spatial-layers", str(SPATIAL_LAYERS),
        "--spatial-heads", str(SPATIAL_HEADS),
        "--valid-actions", VALID_ACTIONS,
        "--lr", str(LR),
        "--ent-coef", str(ENT_COEF),
        "--ent-coef-final", str(ENT_COEF_FINAL),
        "--clip-coef", str(CLIP_COEF),
        "--vf-coef", str(VF_COEF),
        "--max-grad-norm", str(MAX_GRAD_NORM),
        "--gamma", str(GAMMA),
        "--gae-lambda", str(GAE_LAMBDA),
        "--eval-interval", str(EVAL_INTERVAL),
        "--eval-episodes", str(EVAL_EPISODES),
        "--save-interval", str(SAVE_INTERVAL),
        "--log-interval", str(LOG_INTERVAL),
        "--run-name", exp["name"],
    ]
    return cmd


def run_gpu(exp: dict, gpu_id: int):
    """在指定 GPU 上运行训练"""
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
    cmd = build_training_cmd(exp, gpu_id)
    cmd_str = " ".join(str(c) for c in cmd)
    
    print(f"\n{'='*60}")
    print(f"[GPU {gpu_id}] 启动实验: {exp['name']}")
    print(f"[GPU {gpu_id}] 环境: {exp['env_id']}")
    print(f"[GPU {gpu_id}] 总步数: {exp['total_steps']:,}")
    print(f"[GPU {gpu_id}] 并行环境: {exp['num_envs']}")
    print(f"[GPU {gpu_id}] 模型: d_model={exp['d_model']}, layers={exp['mamba_layers']}")
    print(f"{'='*60}")
    
    log_path = WORKDIR / f"train_{exp['name']}_{gpu_id}.log"
    with open(log_path, "w") as log_file:
        proc = subprocess.Popen(
            cmd,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )
        proc.wait()
    
    print(f"[GPU {gpu_id}] 实验完成: {exp['name']} (exit code={proc.returncode})")
    return proc.returncode


# 检测可用 GPU 数量
import torch
num_gpus = torch.cuda.device_count()
print(f"可用 GPU 数量: {num_gpus}")

if num_gpus >= 2:
    print("\n✓ 双 GPU 模式: 两个实验并行执行")
    
    # 并行启动
    start_time = time.time()
    t_a = threading.Thread(target=run_gpu, args=(EXP_A, 0), daemon=False)
    t_b = threading.Thread(target=run_gpu, args=(EXP_B, 1), daemon=False)
    
    t_a.start()
    # 小延迟避免两个进程同时初始化 CUDA
    time.sleep(5)
    t_b.start()
    
    t_a.join()
    t_b.join()
    
    elapsed = time.time() - start_time
    print(f"\n✓ 两个实验完成, 总耗时: {elapsed/3600:.1f} 小时")

else:
    print("\n⚠ 单 GPU 模式: 两个实验串行执行")
    print("  注意: Kaggle 单个 session 最多 12 小时, 5M+10M steps 可能不够")
    
    start_time = time.time()
    run_gpu(EXP_A, 0)
    run_gpu(EXP_B, 0)
    elapsed = time.time() - start_time
    print(f"\n✓ 两个实验完成, 总耗时: {elapsed/3600:.1f} 小时")


# ===== 打印结果摘要 =====
print(f"\n{'='*60}")
print("训练结果摘要")
print(f"{'='*60}")

for exp in [EXP_A, EXP_B]:
    run_dir = RUNS_DIR / exp["name"]
    if run_dir.exists():
        # 检查 eval 日志
        import glob
        tfevents = list(run_dir.glob("events.out.tfevents.*"))
        ckpts = list(run_dir.glob("model_*.pt")) + list(run_dir.glob("model_latest.pt"))
        print(f"\n{exp['name']}:")
        print(f"  环境: {exp['env_id']}")
        print(f"  总步数: {exp['total_steps']:,}")
        print(f"  TensorBoard 事件: {len(tfevents)} 个")
        print(f"  保存的检查点: {len(ckpts)} 个")
        for ckpt in sorted(ckpts):
            size_mb = ckpt.stat().st_size / 1e6
            print(f"    {ckpt.name}: {size_mb:.1f} MB")
    else:
        print(f"\n{exp['name']}: 运行目录不存在 (训练可能失败)")

print(f"\n所有输出保存在: {RUNS_DIR}")


### T4 16GB 极限优化说明

| 参数 | 值 | 原因 |
|------|-----|------|
| `num_envs` | **128** | T4 有 16GB, 单进程 MiniGrid 内存占用极小 (~50MB/128 envs), 极限并行加速 sample collection |
| `num_steps` | **256** | 长 rollout 提升 GAE 优势估计质量, 匹配 `context_len=128` 给 Mamba 足够历史 |
| `d_model` | **256** | Mamba-3 可以在更大 d_model 下高效运行; 更大的 hidden dim 提升记忆容量 |
| `mamba_layers` | **6** | 深层 Mamba 叠加更强的序列建模能力 |
| `batch_chunks` | **64** | 每次 PPO update 打包 64 个 sequence chunk 进行大 batch 训练, 榨干 GPU |
| `context_len` | **128** | 覆盖整个 MemoryS11/S17Random 的走廊长度 |
| `chunk_len` | **128** | 与 context_len 一致, 避免 padding 浪费 |
| `d_state` | **16** | Mamba-1 保守值; 如果升到 Mamba-3 后稳定可用 d_state=64 获得更强记忆 |
| `lr` | **1e-4** | Mamba-3 比 LSTM 对学习率更敏感, 保守学习率更稳 |
| `ent_coef → ent_coef_final` | **0.01 → 0.001** | 前期鼓励探索(找走廊/岔路), 后期退火到 greedy |

### 显存预算 (单 GPU, 128 envs × 256 steps)

| 组件 | 估算 |
|------|------|
| 模型参数 (d_model=256, 6 layers) | ~5M = **20 MB** |
| 优化器状态 (Adam) | **60 MB** |
| Rollout buffer (128×256 步) | **~30 MB** |
| Mamba-1 forward activations | **~2-3 GB** |
| PPO backward + gradients | **~3-5 GB** |
| **总计** | **~6-9 GB** (T4 16GB 余量充足) |

### 如果 OOM

1. 降 `num_envs` 到 64
2. 降 `mamba_layers` 到 4
3. 降 `batch_chunks` 到 32
4. 如果用的是 Mamba-3 (不是 Mamba-1), `d_state=128` 可能需要降到 `d_state=64`

### 预期结果

- **S11**: Mamba-3 应该在 2M-3M 步达到 80%+ success rate; MLP baseline 卡在 ~50%
- **S17Random**: Mamba-3 应该在 5M-8M 步开始显示出记忆优势; MLP 完全失败 (~50%)</cell_id>
